# Classification of Clinical Depression in Students
## Mixed_Group_2 - CS 513 - Spring 2025

#### Group Members
- John Graham (20006581)
- Ramses Peralta (20014942)
- Suprith Reddy (20010383)

#### Dataset Used
Link [here](https://www.kaggle.com/datasets/hopesb/student-depression-dataset)

### Initial Setup

#### Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from os import getcwd

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#### Load Dataset (`data/Student Depression Dataset.csv`)

In [ ]:
RANDOM_STATE = 513

accuracies = {
    "Random Forest": -1,
    "CART": -1,
    "KNN": -1,
    "Naive Bayes": -1,
    "Logistic Regression": -1,
    "ANN": -1,
    "SVM": -1
}

cwd = getcwd()
data = "data"
raw_data_file = "Student Depression Dataset.csv"

df = pd.read_csv(f"{cwd}/{data}/{raw_data_file}", na_values="?").dropna() # drop any null values

df.info()
df.head()

#### Cleaning Data

In [ ]:
df = df[df["Profession"] == "Student"] # filter to only students (few non-students in dataset)
df = df.drop(["id", "Profession", "Work Pressure", "Job Satisfaction"], axis=1) # drop id, profession (not needed for training), work pressure, and job satisfaction (not relevant to students)

# Convert these categorical yes/no questions to 1/0
df["Have you ever had suicidal thoughts ?"] = df["Have you ever had suicidal thoughts ?"].map(lambda x: 1 if x == "Yes" else 0)
df["Family History of Mental Illness"] = df["Family History of Mental Illness"].map(lambda x: 1 if x == "Yes" else 0)

df.info()
df.head()

In [ ]:
df = df[df["Sleep Duration"] != "Others"] # drop "Others" from Sleep Duration
df = df[df["Dietary Habits"] != "Others"] # drop "Others" from Dietary Habits

# Convert Sleep Duration to numerical values
# 1 => Less than 5 hours
# 2 => 5-6 hours
# 3 => 7-8 hours
# 4 => More than 8 hours
df["Sleep Duration"] = df["Sleep Duration"].map(lambda x: 1 if x == "Less than 5 hours" else 2 if x == "5-6 hours" else 3 if x == "7-8 hours" else 4)

# Convert Dietary Habits to numerical values
# 1 => Unhealthy
# 2 => Moderate
# 3 => Healthy
df["Dietary Habits"] = df["Dietary Habits"].map(lambda x: 1 if x == "Unhealthy" else 2 if x == "Moderate" else 3)

df.info()
df.head()

#### Exploratory Data Analysis

In [ ]:
df.describe()

#### Train-Test Split

In [ ]:
attr = df.drop("Depression", axis=1)
target = df["Depression"]

attr_train, attr_test, target_train, target_test = train_test_split(attr, target, test_size=0.2, random_state=RANDOM_STATE)

#### Preprocessor Configuration

In [ ]:
categorical_columns = attr.columns[attr.dtypes == "object"].tolist()
numerical_columns = attr.columns[attr.dtypes != "object"].tolist()
print(f"categorical columns = {categorical_columns}")
print(f"numerical columns = {numerical_columns}")

preprocessor = ColumnTransformer(transformers=[
    ('categorical', OneHotEncoder(handle_unknown="infrequent_if_exist"), categorical_columns),
    ('numerical', MinMaxScaler(), numerical_columns)
])

### Random Forest Classification

#### Model Configuration - Random Forest

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

#### Model Fitting & Predictions - Random Forest

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - Random Forest

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["Random Forest"] = acc
print("Random Forest accuracy: ", accuracies["Random Forest"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
feature_importance = model.named_steps['classifier'].feature_importances_

# grouped by feature names from the preprocessor (since there are more columns due to one-hot encoding)
importances = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values(by='importance', ascending=False)

ax = plt.subplot()
sns.barplot(ax=ax, x='importance', y='feature', data=importances.head(10))
plt.title('10 Most Important Features (RF)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### CART Classification

#### Model Configuration - CART

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier())
])

#### Model Fitting & Predictions - CART

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - CART

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["CART"] = acc
print("CART accuracy: ", accuracies["CART"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### KNN Classification

#### Model Configuration - KNN

In [ ]:
K = 5 # change value of K

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=K))
])

#### Model Fitting & Predictions - KNN

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - KNN

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["KNN"] = acc
print("KNN accuracy: ", accuracies["KNN"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Naive Bayes Classification

#### Model Configuration - Naive Bayes

In [ ]:
nb_preprocessor = ColumnTransformer(transformers=[
    ('categorical', OneHotEncoder(handle_unknown="infrequent_if_exist", sparse_output=False), categorical_columns),
    ('numerical', MinMaxScaler(), numerical_columns)
]) # spare_output = False to resolve errors

model = Pipeline(steps=[
    ('preprocessor', nb_preprocessor),
    ('classifier', GaussianNB()) # gaussian for numerical data
])

#### Model Fitting & Predictions - Naive Bayes

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - Naive Bayes

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["Naive Bayes"] = acc
print("Naive Bayes accuracy: ", accuracies["Naive Bayes"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Logistic Regression Classification

#### Model Configuration - Logistic Regression

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', nb_preprocessor),
    ('classifier', LogisticRegression())
])

#### Model Fitting & Predictions - Logistic Regression

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - Logistic Regression

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["Logistic Regression"] = acc
print("Logistic Regression accuracy: ", accuracies["Logistic Regression"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### ANN Classification

#### Model Configuration - ANN

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', nb_preprocessor),
    ('classifier', MLPClassifier(hidden_layer_sizes=(4,), max_iter=10000, random_state=RANDOM_STATE))
])

#### Model Fitting & Predictions - ANN

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - ANN

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["ANN"] = acc
print("ANN accuracy: ", accuracies["ANN"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
sns.lineplot(x=range(len(model.named_steps["classifier"].loss_curve_)), y=model.named_steps["classifier"].loss_curve_)
plt.title('Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

#### Evaluating Different Parameters for ANN

In [ ]:
ACTIVATION = ['identity', 'logistic', 'tanh']   # default (above) = relu
SOLVER = ['lbfgs', 'sgd', 'adam']               # default (above) = adam
ALPHA = [0.0001, 0.01, 0.1]                     # default (above) = 0.001
HIDDEN_LAYER_SIZES = [(2,), (8,), (16,)]        # default (above) = (4,)

results = [] # array of tuples (param_type, param_val, acc)

for activation in ACTIVATION:
    print(f"activation = {activation}")
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(activation=activation, hidden_layer_sizes=(4,), max_iter=10000, random_state=RANDOM_STATE))
    ])
    model.fit(attr_train, target_train)
    target_pred = model.predict(attr_test)
    acc = accuracy_score(target_test, target_pred)
    results.append(("activation", activation, acc))
    print(f"acc = {acc}")
    print("=====================================")

for solver in SOLVER:
    print(f"solver = {solver}")
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(solver=solver, hidden_layer_sizes=(4,), max_iter=10000, random_state=RANDOM_STATE))
    ])
    model.fit(attr_train, target_train)
    target_pred = model.predict(attr_test)
    acc = accuracy_score(target_test, target_pred)
    results.append(("solver", solver, acc))
    print(f"acc = {acc}")
    print("=====================================")

for alpha in ALPHA:
    print(f"alpha = {alpha}")
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(learning_rate_init=alpha, hidden_layer_sizes=(4,), max_iter=10000, random_state=RANDOM_STATE))
    ])
    model.fit(attr_train, target_train)
    target_pred = model.predict(attr_test)
    acc = accuracy_score(target_test, target_pred)
    results.append(("alpha", alpha, acc))
    print(f"acc = {acc}")
    print("=====================================")

for layers in HIDDEN_LAYER_SIZES:
    print(f"layers = {layers}")
    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MLPClassifier(hidden_layer_sizes=layers, max_iter=10000, random_state=RANDOM_STATE))
    ])
    model.fit(attr_train, target_train)
    target_pred = model.predict(attr_test)
    acc = accuracy_score(target_test, target_pred)
    results.append(("layers", layers, acc))
    print(f"acc = {acc}")
    print("=====================================")

standard_preprocessor = ColumnTransformer(transformers=[
    ('categorical', OneHotEncoder(handle_unknown="infrequent_if_exist"), categorical_columns),
    ('numerical', StandardScaler(), numerical_columns)
])

model = Pipeline(steps=[
    ('preprocessor', standard_preprocessor),
    ('classifier', MLPClassifier(hidden_layer_sizes=(4,), max_iter=10000, random_state=RANDOM_STATE))
])
print("Standard Scaler")
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)
acc = accuracy_score(target_test, target_pred)
results.append(("standard", "Standard Scaler", acc))
print(f"acc = {acc}")
print("=====================================")

results = sorted(results, key=lambda x: x[2], reverse=True)
print("Best Results:")
for ptype, param, acc in results[:5]:
    print(f"param ({ptype}) = {param}, acc = {acc}")

### Support Vector Machine Classification

#### Model Configuration - SVC

In [ ]:
model = Pipeline(steps=[
    ('preprocessor', nb_preprocessor),
    ('classifier', SVC(random_state=RANDOM_STATE))
])

#### Model Fitting & Predictions - SVC

In [ ]:
model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)

#### Metrics & Accuracy - SVC

In [ ]:
acc = accuracy_score(target_test, target_pred)
cm = confusion_matrix(target_test, target_pred)
cr = classification_report(target_test, target_pred)

print(cm)
print(cr)
accuracies["SVM"] = acc
print("SVM accuracy: ", accuracies["SVM"])

In [ ]:
ax = plt.subplot()
sns.heatmap(cm, annot=True, fmt="d", ax=ax, cmap="crest")
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Overall Accuracies

In [ ]:
for i, (k, v) in enumerate(sorted(accuracies.items(), key=lambda x: x[1], reverse=True)):
    print(f"{i + 1}. {k} accuracy: {v:.3f}")

print(f"\nAverage accuracy: {sum(accuracies.values()) / len(accuracies):.3f}")